<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/01_select_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Meet the database & first SELECT

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~30 min**

## Learning objectives
- read data with `SELECT ... FROM`;
- pick columns and rename them with `AS`;
- list unique values with `DISTINCT`;
- peek at a table with `LIMIT` and count rows with `COUNT(*)`.

A **relational database** stores data in linked **tables**. Ours describes a microbial-ecology study in four tables:

| table | one row = | key columns |
|---|---|---|
| `sites` | a sampling site | `site` |
| `samples` | one sample | `sample_id`, `site`, `environment`, `treatment`, `ph`, … |
| `taxa` | one microbial taxon | `taxon_id`, `genus`, `family`, `phylum`, `domain` |
| `abundance` | reads of a taxon in a sample | `sample_id`, `taxon_id`, `count` |

`abundance` links `samples` and `taxa`, and that link is what makes SQL powerful.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## Setup — run this cell first

This connects the notebook to the example database. It works both on your own computer and on Google Colab; just run it (Shift+Enter). You do not need to understand it.

In [1]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.6/397.6 kB 17.3 MB/s eta 0:00:00


Connecting to 'sqlite:///../data/omics.db'

Connected to omics.db — you are ready.


## 1. See a whole table
`SELECT *` means “all columns”. `LIMIT` keeps the output short; use it when exploring.

In [2]:
%%sql
-- every column, first 5 rows of the samples table
SELECT * FROM samples LIMIT 5;



Running query in 'sqlite:///../data/omics.db'

sample_id,site,environment,treatment,ph,temperature_c,depth_cm,collection_date
S001,LEI_SOIL,Soil,Control,5.88,14.5,9.3,2025-04-11
S002,LEI_SOIL,Soil,Control,6.46,13.4,18.1,2025-08-05
S003,LEI_SOIL,Soil,Control,6.09,13.5,19.3,2025-08-25
S004,LEI_SOIL,Soil,Amended,6.17,17.8,None,2025-07-23
S005,LEI_SOIL,Soil,Amended,6.04,19.1,5.6,2025-04-07


In [4]:
%%sql
-- did this myself --
SELECT sample_id, ph
FROM samples
LIMIT 6

Running query in 'sqlite:///../data/omics.db'

sample_id,ph
S001,5.88
S002,6.46
S003,6.09
S004,6.17
S005,6.04
S006,6.01


## 2. Pick specific columns
List the columns you want instead of `*`.

In [5]:
%%sql
SELECT sample_id, environment, treatment, ph
FROM samples
LIMIT 8;

Running query in 'sqlite:///../data/omics.db'

sample_id,environment,treatment,ph
S001,Soil,Control,5.88
S002,Soil,Control,6.46
S003,Soil,Control,6.09
S004,Soil,Amended,6.17
S005,Soil,Amended,6.04
S006,Soil,Amended,6.01
S007,Soil,Control,6.46
S008,Soil,Control,5.71


Another angle: pick different columns (site metadata) and preview a *different* table.

In [6]:
%%sql
-- a different column selection, this time from the sites table
SELECT site, country, latitude
FROM sites
LIMIT 5;

Running query in 'sqlite:///../data/omics.db'

site,country,latitude
LEI_SOIL,Germany,51.34
UPP_SOIL,Sweden,59.86
LEI_LAKE,Germany,51.42
OSL_LAKE,Norway,59.91
LEI_SED,Germany,51.3


## 3. Unique values with DISTINCT
Which environments and phyla exist?

In [11]:
%%sql
SELECT DISTINCT environment FROM samples;

SELECT DISTINCT site FROM samples; --did ti myself here---


Running query in 'sqlite:///../data/omics.db'

site
LEI_SOIL
UPP_SOIL
LEI_LAKE
OSL_LAKE
LEI_SED
CPH_SED


In [12]:
%%sql
SELECT DISTINCT phylum FROM taxa;

Running query in 'sqlite:///../data/omics.db'

phylum
Pseudomonadota
Bacteroidota
Bacillota
Actinomycetota
Acidobacteriota
Cyanobacteriota
Euryarchaeota
Nitrospirota


`DISTINCT` also works across **several columns** — it then returns unique *combinations*.

In [13]:
%%sql
-- unique (domain, phylum) pairs: which phyla sit under each domain?
SELECT DISTINCT domain, phylum
FROM taxa
ORDER BY domain, phylum;

Running query in 'sqlite:///../data/omics.db'

domain,phylum
Archaea,Euryarchaeota
Bacteria,Acidobacteriota
Bacteria,Actinomycetota
Bacteria,Bacillota
Bacteria,Bacteroidota
Bacteria,Cyanobacteriota
Bacteria,Nitrospirota
Bacteria,Pseudomonadota


## 4. Rename columns with AS
`AS` gives a column a friendlier name in the output.

In [15]:
%%sql
SELECT genus AS bacterium, phylum AS lineage
FROM taxa
LIMIT 10;


Running query in 'sqlite:///../data/omics.db'

RuntimeError: (sqlite3.OperationalError) no such column: bacterium
[SQL: SELECT bacterium AS genus
ORDER BY lineage
LIMIT 5;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


`AS` is optional and also names *computed* columns (more in lesson 03).

In [ ]:
%%sql
-- rename site columns; AS makes the header read the way you want
SELECT site AS location, country AS nation
FROM sites;

## 5. Count rows with COUNT(*)

In [20]:
%%sql
SELECT COUNT(*) AS n_samples FROM samples;


Running query in 'sqlite:///../data/omics.db'

RuntimeError: (sqlite3.OperationalError) no such column: taxa
[SQL: SELECT COUNT(taxa) AS totalsites FROM samples;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


`COUNT(*)` counts rows; `COUNT(column)` counts only non-NULL values, a handy way to spot missing data.

In [25]:
%%sql
-- 36 rows exist, but how many actually have a pH recorded?
-- COUNT(*) counts all rows; COUNT(ph) skips NULLs, so the gap = missing pH values
SELECT COUNT(*) AS n_rows, COUNT(ph) AS n_with_ph
FROM samples;

Running query in 'sqlite:///../data/omics.db'

n_rows,n_with_ph
36,33


---
## Exercises

**Exercise.** Show the first 5 rows of the `taxa` table.

In [30]:
%%sql
-- write your query here
-- did it myself, forgot the ; at the end, technically want to see the whole table
SELECT *
FROM taxa
LIMIT 5 ;



Running query in 'sqlite:///../data/omics.db'

taxon_id,genus,family,phylum,domain
T001,Nitrosomonas,Pseudomonadaceae,Pseudomonadota,Bacteria
T002,Pseudomonas,Pseudomonadaceae,Pseudomonadota,Bacteria
T003,Pseudomonas,Pseudomonadaceae,Pseudomonadota,Bacteria
T004,Nitrosomonas,Pseudomonadaceae,Pseudomonadota,Bacteria
T005,Escherichia,Pseudomonadaceae,Pseudomonadota,Bacteria


<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT * FROM taxa LIMIT 5;
```
</details>

**Exercise.** List the distinct `treatment` values in `samples`.

In [31]:
%%sql
-- write your query here
SELECT DISTINCT treatment FROM samples ;

Running query in 'sqlite:///../data/omics.db'

treatment
Control
Amended


<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT DISTINCT treatment FROM samples;
```
</details>

**Exercise.** How many rows are in the `abundance` table? Name the column `n_records`.

In [34]:
%%sql
-- write your query here
-- did it myself, so "table" is the source
SELECT COUNT(*) AS n_records FROM abundance ;

Running query in 'sqlite:///../data/omics.db'

n_records
1304


<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT COUNT(*) AS n_records FROM abundance;
```
</details>

### Recap
- `SELECT columns FROM table` reads data; `*` = all columns.
- `LIMIT n` shows the first n rows; `DISTINCT` removes duplicates.
- `AS` renames a column; `COUNT(*)` counts rows.
- Next: 02 · Filtering rows with WHERE.